# Парсинг

In [136]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time
import pandas as pd
import numpy as np

In [88]:
def click_el(selector):
    global driver
    driver.find_element(By.CSS_SELECTOR, selector).click()
    time.sleep(2)

def get_real_gdp(year):
    global driver
    options = Options()
    driver = webdriver.Chrome(options=options)
    url = 'https://bea.gov/itable/regional-gdp-and-personal-income'
    driver.get(url)
    time.sleep(3)

    click_el('#test > div.panel.panel-itable.panel-default > div > div:nth-child(3) > div:nth-child(1) > a:nth-child(1)')
    click_el('#vertical_container_7001 > div:nth-child(6)')
    click_el('#tabpanel_1_TableId_5 > div > div > span:nth-child(1)')
    click_el('#iTableForm7007 > div.form-group.row.prompt-container > div > select > option:nth-child(2)')
    click_el('#iTableForm7007 > div:nth-child(2) > div > span')
    click_el('#iTableForm7003 > div:nth-child(4) > div > span')
    if year != 2024:
        click_el('#iTableForm7004 > div:nth-child(1) > div > select > option:nth-child(2)')
        click_el(f'#iTableForm7004 > div:nth-child(1) > div > select > option:nth-child({2026-year})')
    click_el('#iTableForm7004 > div:nth-child(4) > div > span')
    time.sleep(15)
    
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')
    
    driver.quit()
    return soup

In [89]:
def real_gdp_to_df(year):
    s = get_real_gdp(year)
    dict_cols ={'County':[], 'Year':[], 'Real_GGP_base_2017':[]}
    stroki = s.find_all('div', class_='dataTables_scrollBody')
    for i in stroki[0].find_all('tr')[1:]:
        county = i.find_all('td', class_='NormalStyle_left p0 dtfc-fixed-left')[0].text.strip()
        name = i.find_all('span', class_='itable-tooltip')[0].text.strip()
        val = i.find_all('td', class_='NormalStyle p0')[0].text.strip()
        if "Real GDP" in name:
            dict_cols['Real_GGP_base_2017'].append(val)
            dict_cols['Year'].append(year)
            dict_cols['County'].append(county)
    return pd.DataFrame(dict_cols)
df_gdp = pd.DataFrame(columns = ['County', 'Year', 'Real_GGP_base_2017'])
for y in [2020,2021,2022,2023,2024]:
    df_gdp = pd.concat([df_gdp, real_gdp_to_df(y)], ignore_index=True)

In [91]:
df_gdp.to_csv('df_gpd.csv')
df_gdp

,County,Year,Real_GGP_base_2017
0,"Autauga, AL",2020,"1,749,999"
1,"Baldwin, AL",2020,"8,110,635"
2,"Barbour, AL",2020,"733,805"
3,"Bibb, AL",2020,"462,167"
4,"Blount, AL",2020,"907,449"
...,...,...,...
15630,"Sweetwater, WY",2024,"3,043,979"
15631,"Teton, WY",2024,"3,741,782"
15632,"Uinta, WY",2024,"859,720"
15633,"Washakie, WY",2024,"344,609"


In [115]:
def get_gdp_structure(year):
    global driver
    options = Options()
    driver = webdriver.Chrome(options=options)
    url = 'https://bea.gov/itable/regional-gdp-and-personal-income'
    driver.get(url)
    time.sleep(3)

    click_el("#test > div.panel.panel-itable.panel-default > div > div:nth-child(3) > div:nth-child(1) > a:nth-child(1)")
    click_el("#vertical_container_7001 > div:nth-child(6)")
    click_el("#tabpanel_1_TableId_5 > div > div > span:nth-child(4)")
    click_el("#iTableForm7007 > div.form-group.row.prompt-container > div > select > option:nth-child(2)")
    click_el("#iTableForm7007 > div:nth-child(2) > div > span")
    click_el('#iTableForm7003 > div:nth-child(2) > div > select > option:nth-child(1)')
    click_el('#iTableForm7003 > div:nth-child(2) > div > select > option:nth-child(7)')
    click_el('#iTableForm7003 > div:nth-child(2) > div > select > option:nth-child(8)')
    click_el('#iTableForm7003 > div:nth-child(2) > div > select > option:nth-child(15)')
    click_el("#iTableForm7003 > div:nth-child(4) > div > span")
    if year != 2024:
        click_el('#iTableForm7004 > div:nth-child(1) > div > select > option:nth-child(2)')
        click_el(f'#iTableForm7004 > div:nth-child(1) > div > select > option:nth-child({2026-year})')
    click_el("#iTableForm7004 > div:nth-child(4) > div > span")
    time.sleep(15)
    
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')
    
    driver.quit()
    return soup

In [116]:
def gdp_structure_to_df(year):
    s = get_gdp_structure(year)
    dict_cols_1 = {'County':[], 'Year':[], 'Construction':[]}
    dict_cols_2 = {'County':[], 'Year':[], 'Manufacturing':[]}
    dict_cols_3 = {'County':[], 'Year':[], 'Finance':[]}

    stroki = s.find_all('div', class_='dataTables_scrollBody')
    for i in stroki[0].find_all('tr')[1:]:
        county = i.find_all('td', class_='NormalStyle_left p0 dtfc-fixed-left')[0].text.strip()
        name = i.find_all('span', class_='itable-tooltip')[0].text.strip()
        val = i.find_all('td', class_='NormalStyle p0')[0].text.strip()
        if 'Construction' in name:
            dict_cols_1['Construction'].append(val)
            dict_cols_1['Year'].append(year)
            dict_cols_1['County'].append(county)
        if 'Manufacturing' in name:
            dict_cols_2['Manufacturing'].append(val)
            dict_cols_2['Year'].append(year)
            dict_cols_2['County'].append(county)
        if 'Finance' in name:
            dict_cols_3['Finance'].append(val)
            dict_cols_3['Year'].append(year)
            dict_cols_3['County'].append(county)
    df1 = pd.DataFrame(dict_cols_1)
    df2 = pd.DataFrame(dict_cols_2)
    df3 = pd.DataFrame(dict_cols_3)
    merged = pd.merge(df1, df2, on=['County', 'Year'])
    merged = pd.merge(merged, df3, on=['County', 'Year'])
    return merged
df_structure_gdp = pd.DataFrame(columns = ['County', 'Year', 'Construction', 'Manufacturing', 'Finance'])
for y in [2020,2021,2022,2023,2024]:
    df_structure_gdp = pd.concat([df_structure_gdp, gdp_structure_to_df(y)], ignore_index=True)

In [117]:
df_structure_gdp.to_csv('df_structure_gdp.csv')
df_structure_gdp

,County,Year,Construction,Manufacturing,Finance
0,"Autauga, AL",2020,"73,458","237,619","280,323"
1,"Baldwin, AL",2020,"590,290","502,777","2,327,274"
2,"Barbour, AL",2020,"16,437","198,154","101,772"
3,"Bibb, AL",2020,"87,503","54,106","55,765"
4,"Blount, AL",2020,"58,161","102,194","221,824"
...,...,...,...,...,...
15630,"Sweetwater, WY",2024,"115,962","524,010","309,178"
15631,"Teton, WY",2024,"263,071","20,009","793,677"
15632,"Uinta, WY",2024,"54,103","49,533","129,076"
15633,"Washakie, WY",2024,"17,566","53,268","67,040"


In [118]:
df_itog = pd.merge(df_gdp, df_structure_gdp, on=['County', 'Year'])
df_itog.to_csv('df_itog.csv')
df_itog

,County,Year,Real_GGP_base_2017,Construction,Manufacturing,Finance
0,"Autauga, AL",2020,"1,749,999","73,458","237,619","280,323"
1,"Baldwin, AL",2020,"8,110,635","590,290","502,777","2,327,274"
2,"Barbour, AL",2020,"733,805","16,437","198,154","101,772"
3,"Bibb, AL",2020,"462,167","87,503","54,106","55,765"
4,"Blount, AL",2020,"907,449","58,161","102,194","221,824"
...,...,...,...,...,...,...
15630,"Sweetwater, WY",2024,"3,043,979","115,962","524,010","309,178"
15631,"Teton, WY",2024,"3,741,782","263,071","20,009","793,677"
15632,"Uinta, WY",2024,"859,720","54,103","49,533","129,076"
15633,"Washakie, WY",2024,"344,609","17,566","53,268","67,040"


# Индекс

In [291]:
def to_num(x):
    if x == '(D)' or x == '(NA)':
        return np.nan
    x = x.replace(',','')
    return int(x)

df_itog = pd.read_csv('df_itog.csv', index_col=0)
    
df_itog['Real_GGP_base_2017'] = df_itog['Real_GGP_base_2017'].apply(to_num)
df_itog['Construction'] = df_itog['Construction'].apply(to_num)
df_itog['Manufacturing'] = df_itog['Manufacturing'].apply(to_num)
df_itog['Finance'] = df_itog['Finance'].apply(to_num)
df_itog['Year'] = df_itog['Year'].apply(int)
df_itog = df_itog.sort_values(['County','Year'])
df_itog

,County,Year,Real_GGP_base_2017,Construction,Manufacturing,Finance
2329,"Abbeville, SC",2020,572511.0,22363.0,186034.0,106419.0
5456,"Abbeville, SC",2021,560996.0,23247.0,174521.0,103721.0
8583,"Abbeville, SC",2022,579250.0,22690.0,171897.0,123610.0
11710,"Abbeville, SC",2023,565099.0,22602.0,174696.0,NaN
14837,"Abbeville, SC",2024,626626.0,24841.0,188676.0,107624.0
...,...,...,...,...,...,...
2440,"Ziebach, SD",2020,38203.0,NaN,NaN,11302.0
5567,"Ziebach, SD",2021,41895.0,NaN,NaN,12864.0
8694,"Ziebach, SD",2022,51732.0,NaN,NaN,NaN
11821,"Ziebach, SD",2023,62287.0,NaN,NaN,NaN


In [292]:
df_itog['Construction_share'] = df_itog['Construction'] / df_itog['Real_GGP_base_2017']
df_itog['Manufacturing_share'] = df_itog['Manufacturing'] / df_itog['Real_GGP_base_2017']
df_itog['Finance_share'] = df_itog['Finance'] / df_itog['Real_GGP_base_2017']
shares = df_itog.groupby('County')[['Construction_share', 'Manufacturing_share','Finance_share']].mean().reset_index()
shares

,County,Construction_share,Manufacturing_share,Finance_share
0,"Abbeville, SC",0.039862,0.308607,0.188979
1,"Acadia, LA",0.060160,0.081528,0.167529
2,"Accomack, VA",0.026495,0.208076,0.142227
3,"Ada, ID",0.058563,0.069346,0.206189
4,"Adair, IA",0.051282,0.121697,0.143088
...,...,...,...,...
3122,"Yuma, AZ",0.035662,0.058686,0.165976
3123,"Yuma, CO",0.033971,0.070777,0.184749
3124,"Zapata, TX",0.020036,0.005812,0.228294
3125,"Zavala, TX",0.019051,0.057281,0.134543


In [293]:
def avg_growth(x):
    gr = []
    for i in range(4):
        pred = x.iloc[i]
        cur = x.iloc[i+1]
        if pd.isna(pred) == False and pd.isna(cur) == False and pred!=0:
            gr.append(cur/pred - 1)
    if len(gr) == 0:
        return np.nan
    return sum(gr)/len(gr)

vr = df_itog.groupby('County')['Manufacturing'].agg(avg_growth).reset_index().rename(columns={'Manufacturing': 'Manufacturing_growth'})
shares = pd.merge(vr, shares, on='County')

vr = df_itog.groupby('County')['Construction'].agg(avg_growth).reset_index().rename(columns={'Construction': 'Construction_growth'})
shares = pd.merge(vr, shares, on='County')

vr = df_itog.groupby('County')['Finance'].agg(avg_growth).reset_index().rename(columns={'Finance': 'Finance_growth'})
shares_growth = pd.merge(vr, shares, on='County')
shares_growth

,County,Finance_growth,Construction_growth,Manufacturing_growth,Construction_share,Manufacturing_share,Finance_share
0,"Abbeville, SC",0.083201,0.027688,0.004846,0.039862,0.308607,0.188979
1,"Acadia, LA",0.016844,-0.010877,0.067941,0.060160,0.081528,0.167529
2,"Accomack, VA",-0.048087,0.063335,0.007442,0.026495,0.208076,0.142227
3,"Ada, ID",0.056023,0.054314,-0.066573,0.058563,0.069346,0.206189
4,"Adair, IA",0.077678,0.103093,-0.099100,0.051282,0.121697,0.143088
...,...,...,...,...,...,...,...
3122,"Yuma, AZ",0.078640,0.058465,0.069018,0.035662,0.058686,0.165976
3123,"Yuma, CO",0.082356,-0.028369,0.281301,0.033971,0.070777,0.184749
3124,"Zapata, TX",-0.060996,0.122802,0.083804,0.020036,0.005812,0.228294
3125,"Zavala, TX",0.090594,NaN,NaN,0.019051,0.057281,0.134543


In [294]:
def w_index(x):
    total = 0
    w_sum = 0
    for s in ['Construction', 'Manufacturing', 'Finance']:
        w, g = x[f'{s}_share'], x[f'{s}_growth']
        if pd.isna(w) == False and pd.isna(g) == False:
            total += w
            w_sum += w*g
    if total > 0:
        return w_sum/total
    return np.nan

shares_growth['index'] = shares_growth.apply(w_index, axis=1)

county_index = shares_growth[['County', 'index']]
county_index = county_index[~county_index['index'].isna()]
county_index.to_csv('county_index.csv')
county_index

,County,index
0,"Abbeville, SC",0.034092
1,"Acadia, LA",0.024923
2,"Accomack, VA",-0.009588
3,"Ada, ID",0.030277
4,"Adair, IA",0.013736
...,...,...
3122,"Yuma, AZ",0.073707
3123,"Yuma, CO",0.118001
3124,"Zapata, TX",-0.043195
3125,"Zavala, TX",0.090594


In [300]:
county_index.sort_values('index')

,County,index
311,"Burke, GA",-0.478603
880,"Esmeralda, NV",-0.238357
1777,"Marion, GA",-0.233061
2685,"Stewart, GA",-0.223119
185,"Benton, MS",-0.210117
...,...,...
304,"Buffalo, SD",1.052869
1708,"Loving, TX",1.060449
1346,"Issaquena, MS",5.941258
1470,"Kenedy, TX",10.425033
